<a href="https://colab.research.google.com/github/guruvittal/Gemini-Enterprise-Agent-Platform_Code_Snippets/blob/main/Agents_on_Gemini_Enterprise_Agent_Platform_Agent_Runtime%2C_Cloud_Run%2C_Observability_%26_Analytics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
%pip install --upgrade --user "google-cloud-aiplatform[agent_engines,adk]"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.4/50.4 kB 690.2 kB/s eta 0:00:00
INFO: pip is looking at multiple versions of opentelemetry-exporter-otlp-proto-http to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of grpcio-status to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 234.2/234.2 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.1/108.1 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.4/323.4 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.4/9.4 MB 36.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 514.9/514.9 kB 30.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.9/44.9 kB 2.6 MB/s eta 0:00:00
  Consider adding this directory to PAT

In [1]:
#@title Login & Authenticate
# @markdown Loading the required libraries
from google.colab import drive
from google.colab import auth
import pandas as pd
import os
auth.authenticate_user()

In [2]:
#@title Global Parameters
import vertexai
PROJECT_ID = "vertexsearch-447722"  # @param {type:"string"}
REGION = "us-central1"          # @param {type:"string"}

vertexai.init(project=PROJECT_ID, location=REGION)

/root/.local/lib/python3.12/site-packages/google/cloud/aiplatform/models.py:52: FutureWarning: Support for google-cloud-storage < 3.0.0 will be removed in a future version of google-cloud-aiplatform. Please upgrade to google-cloud-storage >= 3.0.0.
  from google.cloud.aiplatform.utils import gcs_utils


In [3]:
#@title Define a Simple Agent
from google.adk.agents import Agent
from google.adk.tools import google_search

root_agent = Agent(
    name="google_search_agent",
    model="gemini-2.5-flash",
    instruction="Answer questions using Google Search when needed. Always cite sources.",
    description="Professional search assistant with Google Search capabilities",
    tools=[google_search]
)

In [4]:
#@title Define a Simple App
import vertexai
from vertexai.agent_engines import AdkApp

# Initialize the Agent Platform client with v1beta1 API for agent identity support
client = vertexai.Client(
  project=PROJECT_ID,
  location=REGION,
  http_options=dict(api_version="v1beta1")
)

# Use the proper wrapper class for your Agent Framework
app = AdkApp(agent=root_agent)


In [5]:
#@title Deploy on Agent Run time
from vertexai import types
# Deploy the agent with Agent Identity
remote_app = client.agent_engines.create(
  agent=app,
  config={
    "display_name": "running-agent-with-identity",
    "identity_type": types.IdentityType.AGENT_IDENTITY,
    "requirements": ["google-cloud-aiplatform[adk,agent_engines]"],
    "staging_bucket": f"gs://8thbucket",
  },
)

print(f"Effective Identity: {remote_app.api_resource.spec.effective_identity}")

INFO:vertexai_genai.agentengines:Identified the following requirements: {'pydantic': '2.13.4', 'cloudpickle': '3.1.2', 'google-cloud-aiplatform': '1.160.0'}
INFO:vertexai_genai.agentengines:The following requirements are appended: {'cloudpickle==3.1.2', 'pydantic==2.13.4'}
INFO:vertexai_genai.agentengines:The final list of requirements: ['google-cloud-aiplatform[adk,agent_engines]', 'cloudpickle==3.1.2', 'pydantic==2.13.4']
INFO:vertexai_genai.agentengines:Using bucket 8thbucket
INFO:vertexai_genai.agentengines:Wrote to gs://8thbucket/agent_engine/agent_engine.pkl
INFO:vertexai_genai.agentengines:Writing to gs://8thbucket/agent_engine/requirements.txt
INFO:vertexai_genai.agentengines:Creating in-memory tarfile of extra_packages
INFO:vertexai_genai.agentengines:Writing to gs://8thbucket/agent_engine/dependencies.tar.gz
INFO:vertexai_genai.agentengines:Using agent framework: google-adk
INFO:vertexai_genai.agentengines:View progress and logs at https://console.cloud.google.com/logs/query?

Effective Identity: agents.global.org-887195038651.system.id.goog/resources/aiplatform/projects/36841365232/locations/us-central1/reasoningEngines/6115816275957514240


In [6]:
#@title Remember the Agent Engine ID
import os

# Extract the AGENT_ENGINE_ID from the remote_app.name
# The format is projects/PROJECT_NUMBER/locations/LOCATION/reasoningEngines/AGENT_ENGINE_ID
agent_engine_full_name = remote_app.api_resource.name
AGENT_ENGINE_ID = agent_engine_full_name.split('/')[-1]

# Set the environment variable
os.environ['AGENT_ENGINE_ID'] = AGENT_ENGINE_ID

print(f"AGENT_ENGINE_ID has been updated to: {os.environ.get('AGENT_ENGINE_ID')}")

AGENT_ENGINE_ID has been updated to: 6115816275957514240


In [7]:
#@title Retrieve the Agent Engine
from vertexai.agent_engines import AgentEngine

# Retrieve the deployed agent engine to get its resource details.
# The 'name' of the deployed resource is found in remote_app.api_resource.name.
deployed_agent_engine_client = client.agent_engines.get(name=remote_app.api_resource.name)

# The display name is on the api_resource attribute of the retrieved client.
print(f"Agent Engine retrieved: {deployed_agent_engine_client.api_resource.display_name}")

Agent Engine retrieved: running-agent-with-identity


In [ ]:
#@title Look at logs
!gcloud config set project vertexsearch-447722
!gcloud logging read 'resource.type="cloud_run_revision" AND resource.labels.service_name="googlesearch-cloudrun-agent-with-telemetry"' --limit=3 --format="json" --order="asc"

In [ ]:
#@title Update the Agent Run time
remote_app = client.agent_engines.update(
  name=remote_app.api_resource.name, # Corrected: Use 'name' parameter with the full resource name
  agent=app,
  config={
    "display_name": "Agent-with-Google Search",
    "identity_type": types.IdentityType.AGENT_IDENTITY,
    "requirements": ["google-cloud-aiplatform[adk,agent_engines]"],
    "staging_bucket": f"gs://8thbucket",
  },
)

print(f"Effective Identity: {remote_app.api_resource.spec.effective_identity}")

INFO:vertexai_genai.agentengines:Identified the following requirements: {'google-cloud-aiplatform': '1.153.1', 'pydantic': '2.12.3', 'cloudpickle': '3.1.2'}
INFO:vertexai_genai.agentengines:The following requirements are appended: {'cloudpickle==3.1.2', 'pydantic==2.12.3'}
INFO:vertexai_genai.agentengines:The final list of requirements: ['google-cloud-aiplatform[adk,agent_engines]', 'cloudpickle==3.1.2', 'pydantic==2.12.3']
INFO:vertexai_genai.agentengines:Using bucket 8thbucket
INFO:vertexai_genai.agentengines:Wrote to gs://8thbucket/agent_engine/agent_engine.pkl
INFO:vertexai_genai.agentengines:Writing to gs://8thbucket/agent_engine/requirements.txt
INFO:vertexai_genai.agentengines:Creating in-memory tarfile of extra_packages
INFO:vertexai_genai.agentengines:Writing to gs://8thbucket/agent_engine/dependencies.tar.gz
INFO:vertexai_genai.agentengines:Using agent framework: google-adk
INFO:vertexai_genai.agentengines:View progress and logs at https://console.cloud.google.com/logs/query?

Effective Identity: agents.global.org-887195038651.system.id.goog/resources/aiplatform/projects/36841365232/locations/us-central1/reasoningEngines/7389978728735965184


In [ ]:
#@title Create Sessions & List User Sessions

#print(remote_app)
#remote_app.operation_schemas()
USER_ID = "guru"

session = await remote_app.async_create_session(user_id="USER_ID")

print(session)

response = await remote_app.async_list_sessions(user_id="USER_ID")
print(response)



{'id': '1433609849467305984', 'events': [], 'user_id': 'USER_ID', 'state': {}, 'app_name': '7389978728735965184', 'last_update_time': 1780525652.254534}
{'sessions': [{'events': [], 'userId': 'USER_ID', 'lastUpdateTime': 1780525652.254534, 'id': '1433609849467305984', 'state': {}, 'appName': '7389978728735965184'}, {'events': [], 'userId': 'USER_ID', 'lastUpdateTime': 1780523279.154883, 'id': '4437229325946716160', 'appName': '7389978728735965184', 'state': {}}, {'id': '9089447741020438528', 'appName': '7389978728735965184', 'state': {}, 'events': [], 'userId': 'USER_ID', 'lastUpdateTime': 1780523226.236004}, {'id': '6539284452021895168', 'state': {}, 'appName': '7389978728735965184', 'events': [], 'userId': 'USER_ID', 'lastUpdateTime': 1780523150.725112}]}


In [ ]:
#@title Run some conversations

async for event in remote_app.async_stream_query(
    user_id=USER_ID,
#    session_id=session,  # Optional
    message="What is the exchange rate from US dollars to INR today?",
):
  print(event)
print(event['content']['parts'][0]['text'])


{'model_version': 'gemini-2.5-flash', 'content': {'parts': [{'text': 'As of Wednesday, June 3, 2026, the exchange rate from US dollars to Indian Rupees is approximately 1 USD = ₹95.72 to ₹96.06.\n\nSpecific rates from various sources include:\n*   BookMyForex shows 1 US Dollar equaling 95.72 Indian Rupees as of Thursday, June 4, 2026.\n*   Investing.com reports a current USD/INR exchange rate of 95.755.\n*   Xe.com indicates a mid-market rate of $1 = ₹95.7724 as of 19:12 UTC.\n*   Another listing from Xe.com shows $1 = ₹96.0576 as of 22:32 UTC.'}], 'role': 'model'}, 'grounding_metadata': {'grounding_chunks': [{'web': {'domain': 'bookmyforex.com', 'title': 'bookmyforex.com', 'uri': 'https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQGUsvuNMi6JTTzjxvgjt90HAdBtKeQh1MSZJ4RsKx7DpCC6AVJMiyv3qEG1PMwugSYWTh8M5PpvmwEobJGti6lPUYVj0pYLdOxR4i5e2kjsFIiF_gm5K-GWKEBcC-Kqs5lNnKmBv0B7XGswely_SAvYmcC74w=='}}, {'web': {'domain': 'investing.com', 'title': 'investing.com', 'uri': 'https:/

In [ ]:
#@title Agent Resource Name in full
print(remote_app.api_resource.name)

projects/36841365232/locations/us-central1/reasoningEngines/7389978728735965184


In [ ]:
#@title Enable tracing in Agent Run time

# Update the deployed agent with the new environment variables

client.agent_engines.update(
    name=remote_app.api_resource.name,
    agent=app,

    config={
    "display_name": "Agent-with-Google Search",
    "identity_type": types.IdentityType.AGENT_IDENTITY,
    "requirements": ["google-cloud-aiplatform[adk,agent_engines]"],
    "staging_bucket": f"gs://8thbucket",
    "env_vars": {
            "GOOGLE_CLOUD_AGENT_ENGINE_ENABLE_TELEMETRY": "True",
            "OTEL_SEMCONV_STABILITY_OPT_IN": "http",
            "OTEL_INSTRUMENTATION_GENAI_CAPTURE_MESSAGE_CONTENT": "True"
    }
  },
)

INFO:vertexai_genai.agentengines:Identified the following requirements: {'google-cloud-aiplatform': '1.153.1', 'pydantic': '2.12.3', 'cloudpickle': '3.1.2'}
INFO:vertexai_genai.agentengines:The following requirements are appended: {'cloudpickle==3.1.2', 'pydantic==2.12.3'}
INFO:vertexai_genai.agentengines:The final list of requirements: ['google-cloud-aiplatform[adk,agent_engines]', 'cloudpickle==3.1.2', 'pydantic==2.12.3']
INFO:vertexai_genai.agentengines:Using bucket 8thbucket
INFO:vertexai_genai.agentengines:Wrote to gs://8thbucket/agent_engine/agent_engine.pkl
INFO:vertexai_genai.agentengines:Writing to gs://8thbucket/agent_engine/requirements.txt
INFO:vertexai_genai.agentengines:Creating in-memory tarfile of extra_packages
INFO:vertexai_genai.agentengines:Writing to gs://8thbucket/agent_engine/dependencies.tar.gz
INFO:vertexai_genai.agentengines:Using agent framework: google-adk
INFO:vertexai_genai.agentengines:View progress and logs at https://console.cloud.google.com/logs/query?

AgentEngine(api_resource.name='projects/36841365232/locations/us-central1/reasoningEngines/7389978728735965184')

In [ ]:
#@title Deploy Agent to Gemini Enterprise - Get the Authorization

%%bash

curl -X POST \
   -H "Authorization: Bearer $(gcloud auth print-access-token)" \
   -H "Content-Type: application/json" \
   -H "X-Goog-User-Project: vertexsearch-447722" \
   "https://discoveryengine.googleapis.com/v1alpha/projects/36841365232/locations/global/authorizations?authorizationId=newauth" \
   -d '{
      "name": "projects/36841365232/locations/global/authorizations/newauth",
      "serverSideOauth2": {
         "clientId": "asassa.googleusercontent.com",
         "clientSecret": "asasasas-",
         "authorizationUri": "https://accounts.google.com/o/oauth2/auth?access_type=offline&prompt=consent",
         "tokenUri": "https://oauth2.googleapis.com/token"
      }
   }'

In [ ]:
#@title Deploy Agent to Gemini Enterprise - Create the listing
%%bash

   curl -X POST \
      -H "Authorization: Bearer $(gcloud auth print-access-token)" \
      -H "Content-Type: application/json" \
      -H "X-Goog-User-Project: vertexsearch-447722" \
      "https://discoveryengine.googleapis.com/v1alpha/projects/36841365232/locations/global/collections/default_collection/engines/wendy-s_1778542370208/assistants/default_assistant/agents" \
      -d '{
         "displayName": "Agent with Google Search",
         "description": "Demo Agent",
         "icon": {
            "uri": "ICON_URI"
         },
         "adkAgentDefinition": {
            "provisionedReasoningEngine": {
               "reasoningEngine": "projects/36841365232/locations/us-central1/reasoningEngines/7389978728735965184"
            }
         },
         "authorizationConfig": {
            "toolAuthorizations": [
               "projects/36841365232/locations/global/authorizations/newauth"
            ]
         }
      }'

In [ ]:
#@title Cloud Run Directory Structure

%%bash
mkdir CloudRunAgent
cd CloudRunAgent
mkdir GoogleSearchAgent


In [ ]:
#@title Cloud Run .env file

%%writefile CloudRunAgent/.env

# Google Cloud and Vertex AI configuration
GOOGLE_CLOUD_PROJECT=vertexsearch-447722
GOOGLE_CLOUD_LOCATION=us-central1
GOOGLE_GENAI_USE_VERTEXAI=True

# BigQuery Analytics Configuration
BQ_DATASET_ID=agent_observability_dataset
BQ_TABLE_ID=agent_events
ADK_TRACE_TO_CLOUD=true

Writing CloudRunAgent/.env


In [ ]:
#@title Cloud Run Service Account Authorizations

%%bash

bq mk --location=us-central1 agent_observability_dataset

# Grant Trace permission
gcloud projects add-iam-policy-binding vertexsearch-447722 \
    --member="serviceAccount:36841365232-compute@developer.gserviceaccount.com" \
    --role="roles/cloudtrace.agent" --condition=None

# Grant Logging permission
gcloud projects add-iam-policy-binding vertexsearch-447722 \
    --member="serviceAccount:36841365232-compute@developer.gserviceaccount.com" \
    --role="roles/logging.logWriter" --condition=None

# Grant Metrics permission
gcloud projects add-iam-policy-binding vertexsearch-447722 \
    --member="serviceAccount:36841365232-compute@developer.gserviceaccount.com" \
    --role="roles/monitoring.metricWriter" --condition=None

# Grant Data Editor to write the stream traces
gcloud projects add-iam-policy-binding vertexsearch-447722 \
    --member="serviceAccount:36841365232-compute@developer.gserviceaccount.com" \
    --role="roles/bigquery.dataEditor" --condition=None

# Grant Job User to run metadata/table checks
gcloud projects add-iam-policy-binding vertexsearch-447722 \
    --member="serviceAccount:36841365232-compute@developer.gserviceaccount.com" \
    --role="roles/bigquery.jobUser" --condition=None

gcloud projects add-iam-policy-binding vertexsearch-447722 \
    --member="serviceAccount:36841365232-compute@developer.gserviceaccount.com" \
    --role="roles/bigquery.admin" --condition=None

In [ ]:
#@title Cloud Run Agent File

%%writefile CloudRunAgent/GoogleSearchAgent/agent.py

import os
from pathlib import Path

import google.auth
from dotenv import load_dotenv
from google.adk.agents import Agent
from google.adk.tools import google_search

# Load environment variables from .env file in root directory
root_dir = Path(__file__).parent.parent
dotenv_path = root_dir / ".env"
load_dotenv(dotenv_path=dotenv_path)


# Use default project from credentials if not in .env
_, project_id = google.auth.default()
os.environ.setdefault("GOOGLE_CLOUD_PROJECT", project_id)
os.environ.setdefault("GOOGLE_CLOUD_LOCATION", "global")
os.environ.setdefault("GOOGLE_GENAI_USE_VERTEXAI", "True")


root_agent = Agent(
    name="google_search_agent",
    model="gemini-2.5-flash",
    instruction="Answer questions using Google Search when needed. Always cite sources.",
    description="Professional search assistant with Google Search capabilities",
    tools=[google_search]
)

Overwriting CloudRunAgent/GoogleSearchAgent/agent.py


In [ ]:
#@title Cloud Run Dockerfile

%%writefile CloudRunAgent/Dockerfile
FROM python:3.11-slim

WORKDIR /app

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

RUN adduser --disabled-password --gecos "" myuser

COPY --chown=myuser:myuser . .

USER myuser

ENV PATH="/home/myuser/.local/bin:$PATH"

CMD ["python", "main.py"]

Overwriting CloudRunAgent/Dockerfile


In [ ]:
#@title Cloud Run main.py

%%writefile CloudRunAgent/main.py
import os

import uvicorn
from fastapi import FastAPI
from google.adk.cli.fast_api import get_fast_api_app
from google.adk.plugins.bigquery_agent_analytics_plugin import BigQueryAgentAnalyticsPlugin

# --- ADD THIS LOGIC TO AUTO-TRACE SDK CALLS ---
from opentelemetry import trace
from opentelemetry.instrumentation.google_genai import GoogleGenAiSdkInstrumentor

# Initialize instrumentation for the new Google GenAI SDK
if os.environ.get("ADK_TRACE_TO_CLOUD") == "true":
    GoogleGenAiSdkInstrumentor().instrument()
# -----------------------------------------------

class CustomBigQueryPlugin(BigQueryAgentAnalyticsPlugin):
    def __init__(self, *args, **kwargs):
        project_id = os.getenv("GOOGLE_CLOUD_PROJECT")
        dataset_id = os.getenv("BQ_DATASET_ID")
        table_id = os.getenv("BQ_TABLE_ID", "google_search_agent_logs")
        super().__init__(
            project_id=project_id,
            dataset_id=dataset_id,
            table_id=table_id,
        )


# Get the directory where main.py is located (/app inside the container)
BASE_DIR = os.path.dirname(os.path.abspath(__file__))

# Get the directory where main.py is located
AGENT_DIR = os.path.join(BASE_DIR, "GoogleSearchAgent")

# Example session service URI (e.g., SQLite)
# Note: Use 'sqlite+aiosqlite' instead of 'sqlite' because DatabaseSessionService requires an async driver
#SESSION_SERVICE_URI = "sqlite+aiosqlite:///./sessions.db"
SESSION_SERVICE_URI = "sqlite+aiosqlite:////tmp/sessions.db"

# Example allowed origins for CORS
ALLOWED_ORIGINS = ["http://localhost", "http://localhost:8080", "*"]
# Set web=True if you intend to serve a web interface, False otherwise
SERVE_WEB_INTERFACE = True

# Call the function to get the FastAPI app instance
# Ensure the agent directory name ('capital_agent') matches your agent folder
app: FastAPI = get_fast_api_app(
    agents_dir=AGENT_DIR,
    session_service_uri=SESSION_SERVICE_URI,
    allow_origins=ALLOWED_ORIGINS,
    web=SERVE_WEB_INTERFACE,
    trace_to_cloud=True,
    extra_plugins=["main.CustomBigQueryPlugin"],
)

# You can add more FastAPI routes or configurations below if needed
# Example:
# @app.get("/hello")
# async def read_root():
#     return {"Hello": "World"}

if __name__ == "__main__":
    import uvicorn
    import os
    port = int(os.environ.get("PORT", 8080))
    uvicorn.run("main:app", host="0.0.0.0", port=port)

Overwriting CloudRunAgent/main.py


In [ ]:
#@title Cloud Run requirements

%%writefile CloudRunAgent/requirements.txt
google-adk[otel-gcp]
fastapi
uvicorn
sqlalchemy
aiosqlite
google-cloud-bigquery
google-cloud-bigquery-storage
google-cloud-storage
pyarrow
opentelemetry-exporter-gcp-trace
opentelemetry-resourcedetector-gcp

Overwriting CloudRunAgent/requirements.txt


In [ ]:
#@title Cloud Run __init__ file
%%writefile CloudRunAgent/GoogleSearchAgent/__init__.py

from GoogleSearchAgent.agent import root_agent

__all__ = ["root_agent"]

Overwriting CloudRunAgent/GoogleSearchAgent/__init__.py


In [ ]:
#@title Cloud Run Final Layout

%%bash

echo your-project-directory/
echo ├── GoogleSearchAgent/
echo │   ├── __init__.py
echo │   └── agent.py       # contains `root_agent` definition
echo └── .env
echo └── requirements.txt
echo └── main.py
echo └── Dockerfile

your-project-directory/
├── GoogleSearchAgent/
│ ├── __init__.py
│ └── agent.py
└── .env
└── requirements.txt
└── main.py
└── Dockerfile


In [ ]:
#@title Cloud Run Deployment

%%bash
export GOOGLE_CLOUD_LOCATION="us-central1"
export GOOGLE_CLOUD_PROJECT="vertexsearch-447722"
export GOOGLE_GENAI_USE_VERTEXAI="True"

gcloud run deploy googlesearch-cloudrun-agent \
 --source CloudRunAgent \
 --region $GOOGLE_CLOUD_LOCATION \
 --project $GOOGLE_CLOUD_PROJECT \
 --allow-unauthenticated \
 --set-env-vars="GOOGLE_CLOUD_PROJECT=$GOOGLE_CLOUD_PROJECT,GOOGLE_CLOUD_LOCATION=$GOOGLE_CLOUD_LOCATION,GOOGLE_GENAI_USE_VERTEXAI=$GOOGLE_GENAI_USE_VERTEXAI"



Building using Dockerfile and deploying container to Cloud Run service [googlesearch-cloudrun-agent] in project [vertexsearch-447722] region [us-central1]
Building and deploying...
Validating configuration.........done
Uploading sources........done
Building Container.............................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................done
Setting IAM Policy.............done
Creating Revision......................

In [ ]:
%%bash
export GOOGLE_CLOUD_LOCATION="us-central1"
export GOOGLE_CLOUD_PROJECT="vertexsearch-447722"
export GOOGLE_GENAI_USE_VERTEXAI="True"

gcloud run deploy googlesearch-cloudrun-agent-with-telemetry \
  --region=us-central1 \
  --image=us-central1-docker.pkg.dev/vertexsearch-447722/cloud-run-source-deploy/googlesearch-cloudrun-agent:latest \
  --set-env-vars="ADK_TRACE_TO_CLOUD=true,ADK_LOG_LEVEL=info,GOOGLE_CLOUD_PROJECT=vertexsearch-447722,GOOGLE_CLOUD_LOCATION=us-central1,GOOGLE_GENAI_USE_VERTEXAI=True,OTEL_PYTHON_DISABLED_INSTRUMENTATIONS=all" \
  --allow-unauthenticated




In [ ]:
%%bash
export GOOGLE_CLOUD_LOCATION="us-central1"
export GOOGLE_CLOUD_PROJECT="vertexsearch-447722"
export GOOGLE_GENAI_USE_VERTEXAI="True"

gcloud run deploy googlesearch-cloudrun-agent-with-telemetry \
 --source CloudRunAgent \
 --region $GOOGLE_CLOUD_LOCATION \
 --project $GOOGLE_CLOUD_PROJECT \
 --allow-unauthenticated \
 --set-env-vars="ADK_LOG_LEVEL=info,GOOGLE_CLOUD_PROJECT=$GOOGLE_CLOUD_PROJECT,GOOGLE_CLOUD_LOCATION=$GOOGLE_CLOUD_LOCATION,GOOGLE_GENAI_USE_VERTEXAI=$GOOGLE_GENAI_USE_VERTEXAI"

In [ ]:
%%bash
gcloud run deploy googlesearch-cloudrun-agent-with-telemetry \
 --source CloudRunAgent \
 --region us-central1 \
 --project vertexsearch-447722 \
 --allow-unauthenticated \
 --set-env-vars="GOOGLE_CLOUD_PROJECT=vertexsearch-447722,GOOGLE_CLOUD_LOCATION=us-central1,GOOGLE_GENAI_USE_VERTEXAI=True,GOOGLE_CLOUD_AGENT_ENGINE_ENABLE_TELEMETRY=true,OTEL_SEMCONV_STABILITY_OPT_IN=gen_ai_latest_experimental,ADK_ANALYTICS_ASYNC=True"

Building using Dockerfile and deploying container to Cloud Run service [googlesearch-cloudrun-agent-with-telemetry] in project [vertexsearch-447722] region [us-central1]
Building and deploying...
Validating configuration........done
Uploading sources..........done
Building Container.............................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................

In [ ]:
#@title June 6 2026 Cloud Run Final Deployment
%%bash
cd ./CloudRunAgent
gcloud run deploy google-search-agent-new \
  --source . \
  --project vertexsearch-447722 \
  --region us-central1 \
  --allow-unauthenticated \
  --set-env-vars="GOOGLE_CLOUD_PROJECT=vertexsearch-447722,GOOGLE_CLOUD_LOCATION=us-central1,GOOGLE_GENAI_USE_VERTEXAI=True,BQ_DATASET_ID=agent_observability_dataset,BQ_TABLE_ID=agent_events,ADK_TRACE_TO_CLOUD=true"

Building using Dockerfile and deploying container to Cloud Run service [google-search-agent-new] in project [vertexsearch-447722] region [us-central1]
Building and deploying new service...
Validating configuration........done
Uploading sources........done
Building Container......................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................................

In [ ]:
%%bash
export GOOGLE_CLOUD_LOCATION="us-central1"
export GOOGLE_CLOUD_PROJECT="vertexsearch-447722"

adk deploy cloud_run \
  --project=vertexsearch-447722 \
  --region=us-central1 \
  --trace_to_cloud \
  --main=main:app \
  ./CloudRunAgent/GoogleSearchAgent




In [ ]:
%%bash

gcloud run services update adk-default-service-name \
  --region us-central1 \
  --project vertexsearch-447722 \
  --set-env-vars="ADK_SERVE_UI=True,GOOGLE_GENAI_USE_VERTEXAI=True"

Deploying...
Creating Revision........................................................................................................................................................................done
Routing traffic.....done
Done.
Service [adk-default-service-name] revision [adk-default-service-name-00007-wc4] has been deployed and is serving 100 percent of traffic.
Service URL: https://adk-default-service-name-36841365232.us-central1.run.app


In [ ]:
%%bash
gcloud run services update adk-default-service-name \
  --region us-central1 \
  --project vertexsearch-447722 \
  --remove-env-vars="BIG_QUERY_DATASET_ID,BQ_TABLE_ID"

Deploying...
Creating Revision............................................................................................................................................................................................................................................................done
Routing traffic.....done
Done.
Service [adk-default-service-name] revision [adk-default-service-name-00004-84r] has been deployed and is serving 100 percent of traffic.
Service URL: https://adk-default-service-name-36841365232.us-central1.run.app


In [ ]:
%%bash
curl -X POST https://adk-default-service-name-36841365232.us-central1.run.app/apps/GoogleSearchAgent/users/verified_dev_user/sessions \
     -H "Content-Type: application/json" \
     -d '{
       "sessionId": "live_test_session_001",
       "state": {}
     }'

{"id":"live_test_session_001","appName":"GoogleSearchAgent","userId":"verified_dev_user","state":{},"events":[],"lastUpdateTime":1780761950.5753398}

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100   220  100   148  100    72    390    189 --:--:-- --:--:-- --:--:--   580


In [ ]:
%%bash
curl -X POST https://adk-default-service-name-36841365232.us-central1.run.app/run \
     -H "Content-Type: application/json" \
     -d '{
       "appName": "GoogleSearchAgent",
       "userId": "verified_dev_user",
       "sessionId": "live_test_session_001",
       "newMessage": {
         "role": "user",
         "parts": [{"text": "Who is the Prime Minister of China? What is the title of the topmost leader in China - what is he called?"}]
       }
     }'

Internal Server Error

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100   346  100    21  100   325     17    272  0:00:01  0:00:01 --:--:--   290


In [ ]:
%%bash
curl -X GET https://adk-default-service-name-36841365232.us-central1.run.app/apps/GoogleSearchAgent/users/verified_dev_user/sessions

[{"id":"live_test_session_001","appName":"GoogleSearchAgent","userId":"verified_dev_user","state":{},"events":[],"lastUpdateTime":1780755460.4591746}]

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100   150  100   150    0     0   1339      0 --:--:-- --:--:-- --:--:--  1351


In [ ]:
%%bash

curl -X POST https://adk-default-service-name-36841365232.us-central1.run.app/run \
     -H "Content-Type: application/json" \
     -d '{
       "appName": "GoogleSearchAgent",
       "userId": "verified_dev_user",
       "sessionId": "live_test_session_100",
       "newMessage": {
         "role": "user",
         "parts": [{"text": "What is the capital of France?"}]
       }
     }'

{"detail":"Session not found: live_test_session_100"}

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100   303  100    53  100   250     75    354 --:--:-- --:--:-- --:--:--   429


In [ ]:
%%bash
fuser -k 8000/tcp

 24845

8000/tcp:           


In [ ]:
%%bash

adk run ./CloudRunAgent/GoogleSearchAgent

In [ ]:

import subprocess
from google.colab.output import eval_js

# 1. Fire up the ADK web engine on port 8000 in the background
print("Starting ADK local dashboard server...")
subprocess.Popen(["adk", "web", "./CloudRunAgent/GoogleSearchAgent"])

# 2. Use Colab's native JS engine to generate a secure proxy URL
proxy_url = eval_js("google.colab.kernel.proxyPort(8000)")

print("\n🎉 Your Colab Web UI is fully ready!")
print(f"👉 Click this link to open the interface: {proxy_url}")

Starting ADK local dashboard server...

🎉 Your Colab Web UI is fully ready!
👉 Click this link to open the interface: https://8000-m-s-kkb-use1c2-2ezyml6mnfubn-c.us-east1-2.prod.colab.dev
